In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns

# Machine learnings Libraries
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

# Evaluation Libraries
from sklearn.metrics import *

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))
        
import warnings
warnings.filterwarnings('ignore')

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/playground-series-s5e7/sample_submission.csv
/kaggle/input/playground-series-s5e7/train.csv
/kaggle/input/playground-series-s5e7/test.csv


In [2]:
df= pd.read_csv('/kaggle/input/playground-series-s5e7/train.csv')
df_test= pd.read_csv('/kaggle/input/playground-series-s5e7/test.csv')

In [3]:
df.sample(10)

,id,Time_spent_Alone,Stage_fear,Social_event_attendance,Going_outside,Drained_after_socializing,Friends_circle_size,Post_frequency,Personality
15606,15606,8.0,Yes,1.0,2.0,Yes,3.0,1.0,Introvert
14529,14529,2.0,No,4.0,NaN,No,8.0,NaN,Extrovert
11324,11324,0.0,No,6.0,5.0,No,9.0,8.0,Extrovert
7962,7962,1.0,No,9.0,3.0,No,10.0,9.0,Extrovert
396,396,6.0,Yes,NaN,2.0,Yes,4.0,1.0,Introvert
1319,1319,8.0,Yes,2.0,1.0,No,4.0,NaN,Introvert
18233,18233,4.0,No,10.0,6.0,No,10.0,8.0,Extrovert
15497,15497,0.0,No,8.0,3.0,No,15.0,8.0,Extrovert
3409,3409,10.0,Yes,NaN,1.0,Yes,4.0,2.0,Introvert
17936,17936,8.0,Yes,2.0,2.0,Yes,2.0,3.0,Introvert


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18524 entries, 0 to 18523
Data columns (total 9 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   id                         18524 non-null  int64  
 1   Time_spent_Alone           17334 non-null  float64
 2   Stage_fear                 16631 non-null  object 
 3   Social_event_attendance    17344 non-null  float64
 4   Going_outside              17058 non-null  float64
 5   Drained_after_socializing  17375 non-null  object 
 6   Friends_circle_size        17470 non-null  float64
 7   Post_frequency             17260 non-null  float64
 8   Personality                18524 non-null  object 
dtypes: float64(5), int64(1), object(3)
memory usage: 1.3+ MB


**Insights**:

Datapoints : 18524

Independant Features : 8 (Incl id column)

3 Object & 5 Float Columns

Looks Like there are some null values as well


In [5]:
df.isnull().sum()

id                              0
Time_spent_Alone             1190
Stage_fear                   1893
Social_event_attendance      1180
Going_outside                1466
Drained_after_socializing    1149
Friends_circle_size          1054
Post_frequency               1264
Personality                     0
dtype: int64

In [6]:
# Filling Nulls

cols_to_fillna = ['Time_spent_Alone', 'Stage_fear', 'Social_event_attendance',
                  'Going_outside', 'Drained_after_socializing', 'Friends_circle_size',
                  'Post_frequency']

for col in cols_to_fillna:
    if df[col].dtype == 'object':
        mode_val = df[col].mode().iloc[0]
        df[col] = df[col].fillna(mode_val)
        df_test[col] = df_test[col].fillna(mode_val)
    else:
        mean_val = df[col].mean()
        df[col] = df[col].fillna(mean_val).round(2)
        df_test[col] = df_test[col].fillna(mean_val).round(2)


In [7]:
print('Nulls in df = ',df.isnull().sum().sum())
print('Nulls in df_test = ',df_test.isnull().sum().sum())

Nulls in df =  0
Nulls in df_test =  0


In [8]:
df_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6175 entries, 0 to 6174
Data columns (total 8 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   id                         6175 non-null   int64  
 1   Time_spent_Alone           6175 non-null   float64
 2   Stage_fear                 6175 non-null   object 
 3   Social_event_attendance    6175 non-null   float64
 4   Going_outside              6175 non-null   float64
 5   Drained_after_socializing  6175 non-null   object 
 6   Friends_circle_size        6175 non-null   float64
 7   Post_frequency             6175 non-null   float64
dtypes: float64(5), int64(1), object(2)
memory usage: 386.1+ KB


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18524 entries, 0 to 18523
Data columns (total 9 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   id                         18524 non-null  int64  
 1   Time_spent_Alone           18524 non-null  float64
 2   Stage_fear                 18524 non-null  object 
 3   Social_event_attendance    18524 non-null  float64
 4   Going_outside              18524 non-null  float64
 5   Drained_after_socializing  18524 non-null  object 
 6   Friends_circle_size        18524 non-null  float64
 7   Post_frequency             18524 non-null  float64
 8   Personality                18524 non-null  object 
dtypes: float64(5), int64(1), object(3)
memory usage: 1.3+ MB


In [10]:
# check duplicates
df.duplicated().sum()

0

In [11]:
df.columns

Index(['id', 'Time_spent_Alone', 'Stage_fear', 'Social_event_attendance',
       'Going_outside', 'Drained_after_socializing', 'Friends_circle_size',
       'Post_frequency', 'Personality'],
      dtype='object')

In [12]:
df.sample(10)

,id,Time_spent_Alone,Stage_fear,Social_event_attendance,Going_outside,Drained_after_socializing,Friends_circle_size,Post_frequency,Personality
9714,9714,1.00,No,7.0,6.0,No,12.0,7.0,Extrovert
6344,6344,3.14,No,3.0,6.0,No,7.0,7.0,Extrovert
4874,4874,8.00,No,1.0,2.0,No,1.0,0.0,Introvert
11905,11905,2.00,No,6.0,6.0,No,12.0,8.0,Extrovert
14590,14590,0.00,No,8.0,3.0,No,8.0,3.0,Extrovert
13149,13149,2.00,No,9.0,7.0,No,13.0,5.0,Extrovert
8049,8049,2.00,No,4.0,4.0,No,15.0,6.0,Extrovert
5208,5208,1.00,No,8.0,7.0,No,14.0,7.0,Extrovert
5736,5736,6.00,Yes,3.0,1.0,Yes,3.0,1.0,Introvert
3649,3649,11.00,Yes,2.0,1.0,Yes,1.0,3.0,Introvert


In [13]:
# Label Encoding Object Columns
le = LabelEncoder()

cols = ['Stage_fear','Drained_after_socializing']

for col_name in cols:
    df[col_name] = le.fit_transform(df[col_name])
    df_test[col_name] = le.transform(df_test[col_name])
        


In [14]:
print(df.shape[0])
print(df_test.shape[0])

18524
6175


In [15]:
# Feature Scaling
scaler = StandardScaler()

columns_to_scale = ['Time_spent_Alone', 'Social_event_attendance',
       'Going_outside', 'Friends_circle_size','Post_frequency']

df[columns_to_scale] = scaler.fit_transform(df[columns_to_scale])
df_test[columns_to_scale] = scaler.transform(df_test[columns_to_scale])

In [16]:
df.sample(10)

,id,Time_spent_Alone,Stage_fear,Social_event_attendance,Going_outside,Drained_after_socializing,Friends_circle_size,Post_frequency,Personality
5467,5467,0.640859,1,-1.976402,-0.527468,1,-0.974514,-1.792655,Introvert
3817,3817,-0.735784,0,-0.474981,-0.022220,0,0.244566,-0.713166,Extrovert
662,662,-0.735784,0,0.651084,1.493527,0,1.219831,1.805642,Extrovert
2833,2833,-0.047462,0,0.651084,0.988278,0,1.219831,0.366323,Extrovert
12069,12069,0.296698,0,-0.474981,1.493527,0,0.000750,0.006493,Extrovert
4920,4920,-0.047462,0,0.275729,0.483029,0,-0.730698,-0.713166,Extrovert
4021,4021,1.673342,1,-0.850336,-0.002010,1,-1.949779,-1.432825,Introvert
5321,5321,-0.047462,0,1.026440,-0.022220,0,0.244566,0.006493,Extrovert
3929,3929,-1.079945,0,0.275729,-0.002010,0,0.244566,1.445812,Extrovert
12067,12067,1.329181,1,-1.225691,-0.527468,1,-0.730698,-1.432825,Introvert


In [17]:
# Splitting The Data

X = df.drop(columns = ['id', 'Personality'])
y = df['Personality']
# df_test = df_test.drop(columns = ['id'])

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size = 0.25, random_state = 45, stratify = y)

In [18]:
df_test.shape[0]

6175

In [19]:
rf_model = RandomForestClassifier(n_estimators = 100,class_weight='balanced')

rf_model.fit(X, y)

RandomForestClassifier(class_weight='balanced')

In [20]:
#y_pred= rf_model.predict(df_test[:2548])
y_pred= rf_model.predict(X_test)

In [21]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

   Extrovert       1.00      1.00      1.00      3425
   Introvert       0.99      1.00      1.00      1206

    accuracy                           1.00      4631
   macro avg       1.00      1.00      1.00      4631
weighted avg       1.00      1.00      1.00      4631



In [22]:
# submission = pd.DataFrame({
#     'id': df_test['id'],
#     'Personality': y_pred  
# })

# submission.to_csv('submission.csv', index=False)

In [23]:
# Checking accuracy with split data : X_test
y_pred= rf_model.predict(X_test)
print('Test Accuracy Score = ',accuracy_score(y_pred, y_test))

Test Accuracy Score =  0.9978406391708055


In [24]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from sklearn.metrics import classification_report

# Split
X = df.drop(columns=['id', 'Personality'])
y = df['Personality']
df_test_no_id = df_test.drop(columns=['id'])

le = LabelEncoder()
y_enc = le.fit_transform(y)

X_train, X_test, y_train_enc, y_test_enc = train_test_split(X, y_enc, test_size=0.25, random_state=42)

# Train
model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
model.fit(X, y_enc) # using un-split data

# Predict
y_pred_xgb = model.predict(df_test_no_id)


# Evaluate
# print(classification_report(y_test_enc, y_pred_xgb, target_names=le.classes_))

In [25]:
submission = pd.DataFrame({
    'id': df_test['id'],
    'Personality': le.inverse_transform(y_pred_xgb)
})

submission.to_csv('submission.csv', index=False)